# Week 8 - Notebook 1: Classification Algorithms

**Course**: Introduction to Scientific Programming (CNC-UC)  
**Topic**: Machine Learning Classification with Real-World Data

---

## Learning Objectives

By the end of this notebook, you will:
- Understand binary and multi-class classification
- Implement logistic regression, SVM, decision trees, and random forests
- Compare classification algorithms on real data
- Evaluate models using confusion matrices and classification metrics
- Apply classification to real world datasets

## 1. Setup and Data Loading

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Libraries imported successfully")

## 2. Binary Classification: Breast Cancer Detection

We'll use the **Breast Cancer Wisconsin dataset** - a classic medical classification problem.

**Context**: Features computed from digitized images of breast mass cells. Task: predict malignant vs benign tumors.

**Relevance to Neuroscience**: Similar to classifying diseased vs healthy neural tissue from imaging data.

In [ ]:
# Load breast cancer dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target  # 0 = malignant, 1 = benign
feature_names = cancer.feature_names

# Create DataFrame for easier exploration
df_cancer = pd.DataFrame(X, columns=feature_names)
df_cancer['target'] = y

print(f"Dataset shape: {X.shape}")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"\nClass distribution:")
print(pd.Series(y).value_counts())
print(f"\nTarget names: {cancer.target_names}")

# Display first few rows
df_cancer.head()

In [ ]:
# Explore feature distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

# Plot first 6 features
for idx, feature in enumerate(feature_names[:6]):
    axes[idx].hist(df_cancer[df_cancer['target']==0][feature], 
                   alpha=0.5, label='Malignant', bins=30, color='red')
    axes[idx].hist(df_cancer[df_cancer['target']==1][feature], 
                   alpha=0.5, label='Benign', bins=30, color='green')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()

plt.tight_layout()
plt.suptitle('Feature Distributions by Class', y=1.02, fontsize=14, fontweight='bold')
plt.show()

print("Notice how some features show clear separation between classes!")

### 2.1 Data Preparation

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling - CRITICAL for SVM and KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nFeature scaling applied:")
print(f"Original mean: {X_train.mean(axis=0)[:3]}")
print(f"Scaled mean: {X_train_scaled.mean(axis=0)[:3]}")
print(f"Scaled std: {X_train_scaled.std(axis=0)[:3]}")

### 2.2 Logistic Regression

**Logistic Regression** is a linear model for binary classification. Despite its name, it's a classifier, not a regressor.

**Key Properties**:
- Fast and interpretable
- Works well with linearly separable data
- Provides probability estimates
- Good baseline model

In [ ]:
# Train logistic regression
log_reg = LogisticRegression(random_state=42, max_iter=10000)
log_reg.fit(X_train_scaled, y_train)

# Make predictions
y_pred_lr = log_reg.predict(X_test_scaled)
y_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]  # Probability of benign

# Evaluate
accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

print("Logistic Regression Results:")
print(f"Accuracy:  {accuracy_lr:.4f}")
print(f"Precision: {precision_lr:.4f} (of predicted benign, how many are truly benign)")
print(f"Recall:    {recall_lr:.4f} (of truly benign cases, how many we detected)")
print(f"F1-Score:  {f1_lr:.4f} (harmonic mean of precision and recall)")

# Confusion matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'])
plt.title('Confusion Matrix - Logistic Regression')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print(f"\nTrue Negatives (correctly identified malignant): {cm_lr[0,0]}")
print(f"False Positives (malignant predicted as benign): {cm_lr[0,1]}")
print(f"False Negatives (benign predicted as malignant): {cm_lr[1,0]}")
print(f"True Positives (correctly identified benign): {cm_lr[1,1]}")

### 2.3 Support Vector Machine (SVM)

**SVM** finds the optimal hyperplane that maximizes the margin between classes.

**Key Properties**:
- Effective in high-dimensional spaces
- Memory efficient (uses support vectors only)
- Versatile (different kernels for non-linear boundaries)
- Requires feature scaling

In [ ]:
# Train SVM with RBF kernel
svm_model = SVC(kernel='rbf', random_state=42, probability=True)
svm_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_svm = svm_model.predict(X_test_scaled)
y_proba_svm = svm_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate
accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)

print("SVM Results:")
print(f"Accuracy:  {accuracy_svm:.4f}")
print(f"Precision: {precision_svm:.4f}")
print(f"Recall:    {recall_svm:.4f}")
print(f"F1-Score:  {f1_svm:.4f}")
print(f"\nNumber of support vectors: {len(svm_model.support_)}")

# Confusion matrix
cm_svm = confusion_matrix(y_test, y_pred_svm)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'])
plt.title('Confusion Matrix - SVM')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

### 2.4 Decision Tree

**Decision Trees** learn a series of if-then-else rules from the data.

**Key Properties**:
- Highly interpretable (can visualize the tree)
- No feature scaling required
- Handles non-linear relationships
- Prone to overfitting without pruning

In [ ]:
# Train decision tree (limit depth to avoid overfitting)
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(X_train, y_train)  # Note: no scaling needed for trees

# Make predictions
y_pred_tree = tree_model.predict(X_test)
y_proba_tree = tree_model.predict_proba(X_test)[:, 1]

# Evaluate
accuracy_tree = accuracy_score(y_test, y_pred_tree)
precision_tree = precision_score(y_test, y_pred_tree)
recall_tree = recall_score(y_test, y_pred_tree)
f1_tree = f1_score(y_test, y_pred_tree)

print("Decision Tree Results:")
print(f"Accuracy:  {accuracy_tree:.4f}")
print(f"Precision: {precision_tree:.4f}")
print(f"Recall:    {recall_tree:.4f}")
print(f"F1-Score:  {f1_tree:.4f}")

# Visualize the tree (first few levels)
plt.figure(figsize=(20, 10))
plot_tree(tree_model, 
          feature_names=feature_names,  # Must include ALL feature names
          class_names=['Malignant', 'Benign'],
          filled=True, 
          fontsize=10)
plt.title('Decision Tree Structure (max_depth=4)', fontsize=14, fontweight='bold')
plt.show()

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': tree_model.feature_importances_
}).sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Top 10 Most Important Features (Decision Tree)')
plt.gca().invert_yaxis()
plt.show()

### 2.5 Random Forest

**Random Forest** is an ensemble of decision trees that reduces overfitting.

**Key Properties**:
- More robust than single trees
- Provides feature importance estimates
- Handles large datasets well
- Less interpretable than single trees

In [ ]:
# Train random forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Evaluate
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print("Random Forest Results:")
print(f"Accuracy:  {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall:    {recall_rf:.4f}")
print(f"F1-Score:  {f1_rf:.4f}")
print(f"\nNumber of trees: {rf_model.n_estimators}")

# Feature importance from ensemble
rf_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(rf_importance['feature'], rf_importance['importance'], color='forestgreen')
plt.xlabel('Importance')
plt.title('Top 10 Most Important Features (Random Forest)')
plt.gca().invert_yaxis()
plt.show()

### 2.6 Model Comparison

In [ ]:
# Create comparison dataframe
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'SVM', 'Decision Tree', 'Random Forest'],
    'Accuracy': [accuracy_lr, accuracy_svm, accuracy_tree, accuracy_rf],
    'Precision': [precision_lr, precision_svm, precision_tree, precision_rf],
    'Recall': [recall_lr, recall_svm, recall_tree, recall_rf],
    'F1-Score': [f1_lr, f1_svm, f1_tree, f1_rf]
})

print("\nModel Comparison:")
print(results.to_string(index=False))

# Visualize comparison
results_melted = results.melt(id_vars='Model', var_name='Metric', value_name='Score')
plt.figure(figsize=(12, 6))
sns.barplot(data=results_melted, x='Metric', y='Score', hue='Model')
plt.title('Classification Model Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Score')
plt.ylim(0.9, 1.0)
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 2.7 ROC Curves

In [ ]:
# Calculate ROC curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_proba_svm)
fpr_tree, tpr_tree, _ = roc_curve(y_test, y_proba_tree)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)

# Calculate AUC scores
auc_lr = roc_auc_score(y_test, y_proba_lr)
auc_svm = roc_auc_score(y_test, y_proba_svm)
auc_tree = roc_auc_score(y_test, y_proba_tree)
auc_rf = roc_auc_score(y_test, y_proba_rf)

# Plot ROC curves
plt.figure(figsize=(10, 8))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={auc_lr:.3f})', linewidth=2)
plt.plot(fpr_svm, tpr_svm, label=f'SVM (AUC={auc_svm:.3f})', linewidth=2)
plt.plot(fpr_tree, tpr_tree, label=f'Decision Tree (AUC={auc_tree:.3f})', linewidth=2)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={auc_rf:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves - All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

print("\nAUC Interpretation:")
print("AUC = 1.0: Perfect classifier")
print("AUC = 0.5: Random classifier")
print("AUC > 0.9: Excellent classifier")
print("\nAll our models achieve excellent performance (AUC > 0.95)!")

## 3. Multi-Class Classification: Wine Quality

Now let's tackle a **multi-class problem** with the Wine dataset.

**Context**: Chemical analysis of wines from 3 different cultivars in Italy.

In [ ]:
# Load wine dataset
wine = load_wine()
X_wine = wine.data
y_wine = wine.target  # 3 classes: cultivar 0, 1, 2

print(f"Dataset shape: {X_wine.shape}")
print(f"Number of classes: {len(np.unique(y_wine))}")
print(f"Class names: {wine.target_names}")
print(f"\nClass distribution:")
print(pd.Series(y_wine).value_counts().sort_index())

# Split and scale
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine
)

scaler_w = StandardScaler()
X_train_w_scaled = scaler_w.fit_transform(X_train_w)
X_test_w_scaled = scaler_w.transform(X_test_w)

In [ ]:
# Train multi-class classifiers
models_mc = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

results_mc = {}
for name, model in models_mc.items():
    model.fit(X_train_w_scaled, y_train_w)
    y_pred = model.predict(X_test_w_scaled)
    
    results_mc[name] = {
        'accuracy': accuracy_score(y_test_w, y_pred),
        'predictions': y_pred
    }

# Display results
print("Multi-Class Classification Results:\n")
for name, result in results_mc.items():
    print(f"{name}: Accuracy = {result['accuracy']:.4f}")

In [ ]:
# Detailed classification report for best model
best_model_name = 'Random Forest'
y_pred_best = results_mc[best_model_name]['predictions']

print(f"\nDetailed Report for {best_model_name}:\n")
print(classification_report(y_test_w, y_pred_best, target_names=wine.target_names))

# Confusion matrix for multi-class
cm_mc = confusion_matrix(y_test_w, y_pred_best)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_mc, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=wine.target_names,
            yticklabels=wine.target_names)
plt.title(f'Multi-Class Confusion Matrix - {best_model_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 4. Key Takeaways

### Algorithm Selection Guide:

**Use Logistic Regression when**:
- You need a fast baseline
- Interpretability is important
- Data is roughly linearly separable

**Use SVM when**:
- You have high-dimensional data
- Clear margin of separation exists
- Dataset is not too large

**Use Decision Trees when**:
- Interpretability is critical
- Data has non-linear relationships
- You need feature importance

**Use Random Forest when**:
- You want robust predictions
- You have enough data for ensemble methods
- Some overfitting is acceptable for accuracy

### Best Practices:
1. **Always scale features** for SVM and KNN
2. **Use stratified splits** to maintain class proportions
3. **Compare multiple metrics**, not just accuracy
4. **Check confusion matrices** to understand error types
5. **Use cross-validation** for reliable performance estimates

## 5. Practice Exercises

Try these on your own:

1. **Experiment with K-Nearest Neighbors**: Train a KNN classifier on the breast cancer data. How does it compare to other models?

2. **Feature Selection**: Use only the top 10 most important features from Random Forest. Does performance change?

3. **Hyperparameter Tuning**: Try different values of `max_depth` for the decision tree. What happens to training vs test accuracy?

4. **Imbalanced Classes**: Create an imbalanced version of the dataset (e.g., 90% one class). How do the algorithms perform?

5. **Your Own Data**: Apply these techniques to your neuroscience research data!